# Advanced `UserDict` Problems — With Complete Solutions

This notebook is an advanced practice workbook about Python's `collections.UserDict`, custom mapping behavior, validation, invariants, and the practical differences between subclassing `dict` and subclassing `UserDict`.

## Source-derived core ideas

The source lesson develops these ideas:

- A wrapper around a backing dictionary can enforce custom `__getitem__` / `__setitem__` behavior, but it loses much of the normal dictionary API unless that API is implemented.
- Subclassing the built-in `dict` restores the API, but some built-in methods may bypass Python-level overrides such as `__getitem__` or `__setitem__`.
- `UserDict` is designed to be subclassed and delegates mapping operations in a more predictable, user-defined way.
- `UserDict` stores its underlying ordinary dictionary in `.data`.
- Validation can be centralized in `__setitem__`, which lets construction and operations such as `update` reuse the same rules.
- A `LimitedDict` can constrain the allowed keys and accepted value range.

## Advanced extensions in this notebook

The exercises below go beyond the source lesson and apply production-oriented practices:

- precise exception types,
- rejecting surprising edge cases such as `bool`,
- atomic updates,
- key normalization,
- `__missing__`,
- write-once mappings,
- schema-driven validation,
- public-view vs internal-storage semantics,
- integrity checks for `.data`,
- audit logging,
- copying configured subclasses,
- interoperability,
- and a capstone configuration registry.

All examples use only the Python standard library.


## Best-practices checklist

1. Put invariant enforcement in one central path whenever possible.
2. Use `TypeError` for the wrong kind of object and `ValueError` for a value of the right type that is outside the allowed domain.
3. Remember that `bool` is a subclass of `int`; explicitly reject it when booleans should not count as numeric data.
4. Decide whether transformations happen on write, on read, or both.
5. Keep public semantics consistent across `[]`, `get`, `items`, `update`, construction, copying, and serialization.
6. Test initialization and bulk operations, not only direct assignment.
7. Be careful with `.data`: direct mutation can bypass validation.
8. If a multi-item operation must be all-or-nothing, validate first and commit second.
9. Prefer small reusable validation helpers over duplicated checks.
10. Include assertions for invariants and edge cases.


In [1]:
from collections import UserDict
from collections.abc import Mapping
from numbers import Real
from copy import deepcopy
import math

def expect_exception(exc_type, func, *args, **kwargs):
    # Run func and assert that exc_type is raised.
    try:
        func(*args, **kwargs)
    except exc_type as exc:
        print(f"Expected {exc_type.__name__}: {exc}")
        return exc
    else:
        raise AssertionError(f"Expected {exc_type.__name__} was not raised")

print("Setup complete.")


Setup complete.


## Problem 1 — Trace method dispatch: `dict` vs `UserDict`

Build two mapping subclasses, one from `dict` and one from `UserDict`. Log calls to `__getitem__` and `__setitem__`.

Compare:

- direct assignment,
- direct lookup,
- `get`,
- `update`,
- `dict(mapping)`.

Explain why the difference matters when validation or transformation is implemented in overrides.


### Solution 1


In [2]:
class TraceBuiltInDict(dict):
    def __init__(self, *args, **kwargs):
        self.log = []
        super().__init__(*args, **kwargs)

    def __getitem__(self, key):
        self.log.append(("get", key))
        return super().__getitem__(key)

    def __setitem__(self, key, value):
        self.log.append(("set", key, value))
        return super().__setitem__(key, value)


class TraceUserDict(UserDict):
    def __init__(self, *args, **kwargs):
        self.log = []
        super().__init__(*args, **kwargs)

    def __getitem__(self, key):
        self.log.append(("get", key))
        return super().__getitem__(key)

    def __setitem__(self, key, value):
        self.log.append(("set", key, value))
        return super().__setitem__(key, value)


b = TraceBuiltInDict()
b["x"] = 10
_ = b["x"]
_ = b.get("x")
b.update({"y": 20})
built_in_copy = dict(b)

u = TraceUserDict()
u["x"] = 10
_ = u["x"]
_ = u.get("x")
u.update({"y": 20})
user_copy = dict(u)

print("dict-subclass log:", b.log)
print("UserDict log:", u.log)
print("dict(b):", built_in_copy)
print("dict(u):", user_copy)

assert ("set", "x", 10) in b.log
assert ("set", "x", 10) in u.log
assert ("set", "y", 20) in u.log
assert ("get", "x") in u.log


dict-subclass log: [('set', 'x', 10), ('get', 'x')]
UserDict log: [('set', 'x', 10), ('get', 'x'), ('get', 'x'), ('set', 'y', 20), ('get', 'x'), ('get', 'y')]
dict(b): {'x': 10, 'y': 20}
dict(u): {'x': 10, 'y': 20}


## Problem 2 — Robust integer-view mapping

Implement `StrictIntViewDict(UserDict)`:

- values must be real numbers,
- `bool` must be rejected,
- values must be finite,
- raw real values are stored,
- public reads return truncated integers.

Test direct assignment, `get`, construction, `update`, `.data`, invalid strings, `True`, infinity, and NaN.


### Solution 2


In [3]:
class StrictIntViewDict(UserDict):
    @staticmethod
    def _validate_value(value):
        if isinstance(value, bool) or not isinstance(value, Real):
            raise TypeError("value must be a non-boolean real number")
        if not math.isfinite(float(value)):
            raise ValueError("value must be finite")

    def __setitem__(self, key, value):
        self._validate_value(value)
        super().__setitem__(key, value)

    def __getitem__(self, key):
        return int(super().__getitem__(key))


d = StrictIntViewDict(a=10.9, b=-3.7)
d["c"] = 8.99
d.update({"d": 4.2})

assert d["a"] == 10
assert d.get("b") == -3
assert d["c"] == 8
assert d["d"] == 4

print("Public mapping:", dict(d))
print("Internal .data:", d.data)

assert d.data["a"] == 10.9
assert d["a"] == 10

expect_exception(TypeError, d.__setitem__, "bad", "python")
expect_exception(TypeError, d.__setitem__, "bad", True)
expect_exception(ValueError, d.__setitem__, "bad", float("inf"))
expect_exception(ValueError, d.__setitem__, "bad", float("nan"))


Public mapping: {'a': 10, 'b': -3, 'c': 8, 'd': 4}
Internal .data: {'a': 10.9, 'b': -3.7, 'c': 8.99, 'd': 4.2}
Expected TypeError: value must be a non-boolean real number
Expected TypeError: value must be a non-boolean real number
Expected ValueError: value must be finite
Expected ValueError: value must be finite


ValueError('value must be finite')

The raw value and public value can differ intentionally. The class stores the original real value but defines a transformed public read interface.


## Problem 3 — Prove constructor and `update` reuse validation

Create `PositiveScoreDict(UserDict)`:

- string keys only,
- integer values only,
- reject `bool`,
- range `0..100`,
- count how many times `__setitem__` performs validation.

Verify validation during construction, keyword initialization, direct assignment, and `update`.


### Solution 3


In [4]:
class PositiveScoreDict(UserDict):
    def __init__(self, *args, **kwargs):
        self.validation_calls = 0
        super().__init__(*args, **kwargs)

    def __setitem__(self, key, value):
        self.validation_calls += 1

        if not isinstance(key, str):
            raise TypeError("key must be a string")
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("score must be an integer")
        if not 0 <= value <= 100:
            raise ValueError("score must be between 0 and 100")

        super().__setitem__(key, value)


scores = PositiveScoreDict({"Ada": 98}, Grace=95)
scores["Linus"] = 91
scores.update({"Guido": 99, "Barbara": 97})

print(scores)
print("Validation calls:", scores.validation_calls)

assert scores.validation_calls == 5
expect_exception(ValueError, PositiveScoreDict, {"Ada": 120})
expect_exception(TypeError, PositiveScoreDict, {"Ada": True})
expect_exception(TypeError, PositiveScoreDict, {42: 80})


{'Ada': 98, 'Grace': 95, 'Linus': 91, 'Guido': 99, 'Barbara': 97}
Validation calls: 5
Expected ValueError: score must be between 0 and 100
Expected TypeError: score must be an integer
Expected TypeError: key must be a string


TypeError('key must be a string')

## Problem 4 — Production-grade `LimitedDict`

Improve the source-style `LimitedDict`:

- copy allowed keys into an immutable `frozenset`,
- validate bounds,
- reject `bool`,
- use `KeyError`, `TypeError`, and `ValueError` appropriately,
- expose configuration properties.

Model an RGB dictionary with values `0..255`.


### Solution 4


In [5]:
class LimitedDict(UserDict):
    def __init__(self, keyset, min_value, max_value, *args, **kwargs):
        if isinstance(min_value, bool) or not isinstance(min_value, int):
            raise TypeError("min_value must be an integer")
        if isinstance(max_value, bool) or not isinstance(max_value, int):
            raise TypeError("max_value must be an integer")
        if min_value > max_value:
            raise ValueError("min_value cannot exceed max_value")

        self._keyset = frozenset(keyset)
        self._min_value = min_value
        self._max_value = max_value
        super().__init__(*args, **kwargs)

    @property
    def allowed_keys(self):
        return self._keyset

    @property
    def min_value(self):
        return self._min_value

    @property
    def max_value(self):
        return self._max_value

    def __setitem__(self, key, value):
        if key not in self._keyset:
            raise KeyError(f"invalid key: {key!r}")
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("value must be an integer")
        if not self._min_value <= value <= self._max_value:
            raise ValueError(
                f"value must be between {self._min_value} and {self._max_value}"
            )
        super().__setitem__(key, value)


rgb = LimitedDict(
    {"red", "green", "blue"},
    0,
    255,
    red=12,
    green=128,
    blue=250,
)

rgb["red"] = 200
print(rgb)

assert rgb == {"red": 200, "green": 128, "blue": 250}

expect_exception(KeyError, rgb.__setitem__, "alpha", 255)
expect_exception(TypeError, rgb.__setitem__, "red", 12.5)
expect_exception(TypeError, rgb.__setitem__, "red", True)
expect_exception(ValueError, rgb.__setitem__, "red", 300)
expect_exception(ValueError, LimitedDict, {"x"}, 10, 0)


{'red': 200, 'green': 128, 'blue': 250}
Expected KeyError: "invalid key: 'alpha'"
Expected TypeError: value must be an integer
Expected TypeError: value must be an integer
Expected ValueError: value must be between 0 and 255
Expected ValueError: min_value cannot exceed max_value


ValueError('min_value cannot exceed max_value')

## Problem 5 — Numeric edge cases

Investigate:

```python
True
1
1.0
3 + 0j
float("nan")
float("inf")
```

Then write `validate_measurement(value)` that accepts finite real numbers but rejects booleans.


### Solution 5


In [6]:
values = [True, 1, 1.0, 3 + 0j, float("nan"), float("inf")]

for value in values:
    print(
        repr(value).ljust(12),
        "int?", isinstance(value, int),
        "Real?", isinstance(value, Real),
    )


def validate_measurement(value):
    if isinstance(value, bool) or not isinstance(value, Real):
        raise TypeError("measurement must be a non-boolean real number")
    if not math.isfinite(float(value)):
        raise ValueError("measurement must be finite")
    return value


assert validate_measurement(1) == 1
assert validate_measurement(2.5) == 2.5
expect_exception(TypeError, validate_measurement, True)
expect_exception(TypeError, validate_measurement, 3 + 0j)
expect_exception(ValueError, validate_measurement, float("nan"))
expect_exception(ValueError, validate_measurement, float("inf"))


True         int? True Real? True
1            int? True Real? True
1.0          int? False Real? True
(3+0j)       int? False Real? False
nan          int? False Real? True
inf          int? False Real? True
Expected TypeError: measurement must be a non-boolean real number
Expected TypeError: measurement must be a non-boolean real number
Expected ValueError: measurement must be finite
Expected ValueError: measurement must be finite


ValueError('measurement must be finite')

## Problem 6 — Make `update` atomic

A normal iterative update can partially mutate the object before a later item fails validation.

Implement `AtomicLimitedDict` so that:

1. all candidate entries are collected,
2. all are validated,
3. only then are they committed.

Support mappings, iterables of pairs, and keyword arguments.


### Solution 6


In [7]:
class AtomicLimitedDict(UserDict):
    def __init__(self, keyset, min_value, max_value, *args, **kwargs):
        self._keyset = frozenset(keyset)
        self._min_value = min_value
        self._max_value = max_value
        super().__init__()
        if args or kwargs:
            self.update(*args, **kwargs)

    def _validate_pair(self, key, value):
        if key not in self._keyset:
            raise KeyError(f"invalid key: {key!r}")
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("value must be an integer")
        if not self._min_value <= value <= self._max_value:
            raise ValueError(
                f"value must be between {self._min_value} and {self._max_value}"
            )

    def __setitem__(self, key, value):
        self._validate_pair(key, value)
        super().__setitem__(key, value)

    def update(self, other=None, **kwargs):
        staged = {}

        if other is not None:
            if isinstance(other, Mapping) or hasattr(other, "keys"):
                for key in other.keys():
                    staged[key] = other[key]
            else:
                for key, value in other:
                    staged[key] = value

        staged.update(kwargs)

        for key, value in staged.items():
            self._validate_pair(key, value)

        for key, value in staged.items():
            super().__setitem__(key, value)


atomic_rgb = AtomicLimitedDict(
    {"red", "green", "blue"}, 0, 255,
    red=10, green=20, blue=30
)

before = dict(atomic_rgb)

expect_exception(
    ValueError,
    atomic_rgb.update,
    {"red": 100, "green": 200, "blue": 999},
)

assert dict(atomic_rgb) == before

atomic_rgb.update({"red": 100, "green": 200}, blue=250)
assert atomic_rgb == {"red": 100, "green": 200, "blue": 250}
print(atomic_rgb)


Expected ValueError: value must be between 0 and 255
{'red': 100, 'green': 200, 'blue': 250}


## Problem 7 — Case-insensitive keys with collision-safe normalization

Implement `CaseInsensitiveDict(UserDict)`:

- string keys only,
- storage uses `.casefold()`,
- lookup and containment are case-insensitive,
- preserve most recent original spelling for display,
- reject collisions within one bulk update such as `Host` + `HOST`.


### Solution 7


In [8]:
class CaseInsensitiveDict(UserDict):
    def __init__(self, *args, **kwargs):
        self._original_keys = {}
        super().__init__()
        if args or kwargs:
            self.update(*args, **kwargs)

    @staticmethod
    def _normalize(key):
        if not isinstance(key, str):
            raise TypeError("keys must be strings")
        return key.casefold()

    def __setitem__(self, key, value):
        normalized = self._normalize(key)
        self._original_keys[normalized] = key
        self.data[normalized] = value

    def __getitem__(self, key):
        return self.data[self._normalize(key)]

    def __delitem__(self, key):
        normalized = self._normalize(key)
        del self.data[normalized]
        self._original_keys.pop(normalized, None)

    def __contains__(self, key):
        try:
            normalized = self._normalize(key)
        except TypeError:
            return False
        return normalized in self.data

    def display_items(self):
        return [
            (self._original_keys[norm], value)
            for norm, value in self.data.items()
        ]

    def update(self, other=None, **kwargs):
        pairs = []

        if other is not None:
            if isinstance(other, Mapping) or hasattr(other, "keys"):
                pairs.extend((key, other[key]) for key in other.keys())
            else:
                pairs.extend(other)

        pairs.extend(kwargs.items())

        seen = set()
        staged = []

        for key, value in pairs:
            normalized = self._normalize(key)
            if normalized in seen:
                raise ValueError(
                    f"duplicate normalized key in one update: {key!r}"
                )
            seen.add(normalized)
            staged.append((key, value))

        for key, value in staged:
            self[key] = value


headers = CaseInsensitiveDict({"Content-Type": "application/json"})
assert headers["content-type"] == "application/json"
assert headers["CONTENT-TYPE"] == "application/json"
assert "CoNtEnT-TyPe" in headers

headers["HOST"] = "example.com"
headers["host"] = "api.example.com"

print("Internal:", headers.data)
print("Display:", headers.display_items())

assert headers["Host"] == "api.example.com"
expect_exception(
    ValueError,
    headers.update,
    [("Accept", "json"), ("ACCEPT", "text")],
)
expect_exception(TypeError, headers.__setitem__, 123, "bad")


Internal: {'content-type': 'application/json', 'host': 'api.example.com'}
Display: [('Content-Type', 'application/json'), ('host', 'api.example.com')]
Expected ValueError: duplicate normalized key in one update: 'ACCEPT'
Expected TypeError: keys must be strings


TypeError('keys must be strings')

## Problem 8 — `__missing__` with validation

Build `CountingDict(UserDict)`:

- non-negative integer counts only,
- a missing key should be inserted with `0`,
- then `counts[word] += 1` should work naturally.

Use `__missing__`.


### Solution 8


In [9]:
class CountingDict(UserDict):
    def __setitem__(self, key, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("count must be an integer")
        if value < 0:
            raise ValueError("count cannot be negative")
        super().__setitem__(key, value)

    def __missing__(self, key):
        self[key] = 0
        return 0


counts = CountingDict()

for word in "to be or not to be".split():
    counts[word] += 1

print(counts)

assert counts == {"to": 2, "be": 2, "or": 1, "not": 1}
assert counts["unknown"] == 0
assert "unknown" in counts

expect_exception(ValueError, counts.__setitem__, "bad", -1)


{'to': 2, 'be': 2, 'or': 1, 'not': 1}
Expected ValueError: count cannot be negative


ValueError('count cannot be negative')

## Problem 9 — Write-once keys

Create `WriteOnceDict(UserDict)`:

- a key can be assigned only once,
- deletion is disabled,
- construction and `update` must honor the rule.

Then deliberately mutate `.data` to demonstrate the escape hatch.


### Solution 9


In [10]:
class WriteOnceDict(UserDict):
    def __setitem__(self, key, value):
        if key in self.data:
            raise KeyError(f"key {key!r} is write-once and already exists")
        super().__setitem__(key, value)

    def __delitem__(self, key):
        raise TypeError("deletion is disabled for WriteOnceDict")


settings = WriteOnceDict({"api_version": 1})
settings["region"] = "eu"

expect_exception(KeyError, settings.__setitem__, "api_version", 2)
expect_exception(TypeError, settings.__delitem__, "region")
expect_exception(KeyError, settings.update, {"region": "us"})

settings.data["api_version"] = 999
print("Bypassed invariant through .data:", settings)
assert settings["api_version"] == 999


Expected KeyError: "key 'api_version' is write-once and already exists"
Expected TypeError: deletion is disabled for WriteOnceDict
Expected KeyError: "key 'region' is write-once and already exists"
Bypassed invariant through .data: {'api_version': 999, 'region': 'eu'}


`UserDict.data` is intentionally accessible. If strong encapsulation is required, a custom `MutableMapping` with private backing storage may be a better design. With `UserDict`, treat `.data` as an implementation-level escape hatch.


## Problem 10 — Schema-driven validation and coercion

Build `SchemaDict(UserDict)` where each key has a validator/coercer callable.

Schema:

- `host`: stripped non-empty string
- `port`: integer `1..65535`
- `debug`: exact `bool`
- `timeout`: positive finite real stored as `float`

Unknown keys are rejected.


### Solution 10


In [11]:
def as_nonempty_string(value):
    if not isinstance(value, str):
        raise TypeError("expected a string")
    value = value.strip()
    if not value:
        raise ValueError("string cannot be empty")
    return value


def as_port(value):
    if isinstance(value, bool) or not isinstance(value, int):
        raise TypeError("port must be an integer")
    if not 1 <= value <= 65535:
        raise ValueError("port must be between 1 and 65535")
    return value


def as_bool(value):
    if type(value) is not bool:
        raise TypeError("debug must be exactly bool")
    return value


def as_positive_timeout(value):
    if isinstance(value, bool) or not isinstance(value, Real):
        raise TypeError("timeout must be a real number")
    value = float(value)
    if not math.isfinite(value) or value <= 0:
        raise ValueError("timeout must be positive and finite")
    return value


class SchemaDict(UserDict):
    def __init__(self, schema, *args, **kwargs):
        self._schema = dict(schema)
        super().__init__(*args, **kwargs)

    def __setitem__(self, key, value):
        try:
            validator = self._schema[key]
        except KeyError:
            raise KeyError(f"unsupported field: {key!r}") from None

        coerced = validator(value)
        super().__setitem__(key, coerced)


schema = {
    "host": as_nonempty_string,
    "port": as_port,
    "debug": as_bool,
    "timeout": as_positive_timeout,
}

config = SchemaDict(
    schema,
    host="  example.com  ",
    port=443,
    debug=False,
    timeout=2,
)

print(config)

assert config["host"] == "example.com"
assert config["timeout"] == 2.0

expect_exception(KeyError, config.__setitem__, "username", "ada")
expect_exception(ValueError, config.__setitem__, "port", 70000)
expect_exception(TypeError, config.__setitem__, "debug", 1)
expect_exception(ValueError, config.__setitem__, "host", "   ")


{'host': 'example.com', 'port': 443, 'debug': False, 'timeout': 2.0}
Expected KeyError: "unsupported field: 'username'"
Expected ValueError: port must be between 1 and 65535
Expected TypeError: debug must be exactly bool
Expected ValueError: string cannot be empty


ValueError('string cannot be empty')

## Problem 11 — Separate write transformation from read transformation

Implement `TemperatureDict(UserDict)`:

- input values are Celsius,
- internally store normalized `float` Celsius,
- public reads return Fahrenheit rounded to one decimal place,
- `celsius(key)` returns raw Celsius,
- reject booleans and non-finite values.

Compare `[]`, `get`, `dict(mapping)`, and `.data`.


### Solution 11


In [12]:
class TemperatureDict(UserDict):
    @staticmethod
    def _as_celsius(value):
        if isinstance(value, bool) or not isinstance(value, Real):
            raise TypeError("temperature must be a real number")
        value = float(value)
        if not math.isfinite(value):
            raise ValueError("temperature must be finite")
        return value

    def __setitem__(self, key, value):
        super().__setitem__(key, self._as_celsius(value))

    def __getitem__(self, key):
        c = super().__getitem__(key)
        return round(c * 9 / 5 + 32, 1)

    def celsius(self, key):
        return self.data[key]


temps = TemperatureDict(room=20, freezer=-18.5)

print("Public room:", temps["room"])
print("Public freezer:", temps.get("freezer"))
print("dict(temps):", dict(temps))
print("Internal Celsius:", temps.data)

assert temps["room"] == 68.0
assert temps.celsius("room") == 20.0


Public room: 68.0
Public freezer: -1.3
dict(temps): {'room': 68.0, 'freezer': -1.3}
Internal Celsius: {'room': 20.0, 'freezer': -18.5}


## Problem 12 — Detect invariant corruption through `.data`

Create `IntegrityCheckedScoreDict(UserDict)` with values in `0..100`.

Add:

- `_validate_pair`,
- `validate_integrity()`,
- `replace_data(mapping)` that validates everything before replacement.

Demonstrate direct `.data` corruption and recovery.


### Solution 12


In [13]:
class IntegrityCheckedScoreDict(UserDict):
    def _validate_pair(self, key, value):
        if not isinstance(key, str):
            raise TypeError("key must be a string")
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("score must be an integer")
        if not 0 <= value <= 100:
            raise ValueError("score must be between 0 and 100")

    def __setitem__(self, key, value):
        self._validate_pair(key, value)
        super().__setitem__(key, value)

    def validate_integrity(self):
        for key, value in self.data.items():
            self._validate_pair(key, value)
        return True

    def replace_data(self, mapping):
        staged = dict(mapping)

        for key, value in staged.items():
            self._validate_pair(key, value)

        self.data.clear()
        self.data.update(staged)


scores2 = IntegrityCheckedScoreDict(Ada=99, Grace=98)
assert scores2.validate_integrity()

scores2.data["Ada"] = 999
expect_exception(ValueError, scores2.validate_integrity)

before = dict(scores2.data)
expect_exception(
    ValueError,
    scores2.replace_data,
    {"Ada": 90, "Grace": 500},
)
assert scores2.data == before

scores2.replace_data({"Ada": 100, "Grace": 100})
assert scores2.validate_integrity()
print(scores2)


Expected ValueError: score must be between 0 and 100
Expected ValueError: score must be between 0 and 100
{'Ada': 100, 'Grace': 100}


## Problem 13 — Audit every mutation

Implement `AuditDict(UserDict)`.

Every successful mutation records:

```python
(operation, key, old_value, new_value)
```

Use operations `"set"` and `"delete"`, and a unique sentinel for missing values.


### Solution 13


In [14]:
_MISSING = object()

class AuditDict(UserDict):
    def __init__(self, *args, **kwargs):
        self.audit_log = []
        super().__init__(*args, **kwargs)

    def __setitem__(self, key, value):
        old_value = self.data.get(key, _MISSING)
        super().__setitem__(key, value)
        self.audit_log.append(("set", key, old_value, value))

    def __delitem__(self, key):
        old_value = self.data[key]
        super().__delitem__(key)
        self.audit_log.append(("delete", key, old_value, _MISSING))


audit = AuditDict(a=1)
audit["a"] = 2
audit["b"] = 3
del audit["b"]

for operation, key, old, new in audit.audit_log:
    old_display = "<MISSING>" if old is _MISSING else old
    new_display = "<MISSING>" if new is _MISSING else new
    print(operation, key, old_display, "->", new_display)

before_len = len(audit.audit_log)
expect_exception(KeyError, audit.__delitem__, "does-not-exist")
assert len(audit.audit_log) == before_len


set a <MISSING> -> 1
set a 1 -> 2
set b <MISSING> -> 3
delete b 3 -> <MISSING>
Expected KeyError: 'does-not-exist'


## Problem 14 — Copy configured mappings correctly

Create a bounded mapping with constructor configuration and a custom `copy()` method.

The copy must:

- have the same subclass,
- preserve configuration,
- contain the same data,
- use a different `.data` object.


### Solution 14


In [15]:
class ConfiguredLimitedDict(UserDict):
    def __init__(self, keyset, min_value, max_value, *args, **kwargs):
        self._keyset = frozenset(keyset)
        self._min_value = min_value
        self._max_value = max_value
        super().__init__(*args, **kwargs)

    def __setitem__(self, key, value):
        if key not in self._keyset:
            raise KeyError(f"invalid key: {key!r}")
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("value must be an integer")
        if not self._min_value <= value <= self._max_value:
            raise ValueError("value outside configured bounds")
        super().__setitem__(key, value)

    def copy(self):
        return type(self)(
            self._keyset,
            self._min_value,
            self._max_value,
            self.data,
        )


original = ConfiguredLimitedDict({"x", "y"}, 0, 10, x=1, y=2)
cloned = original.copy()

assert type(cloned) is ConfiguredLimitedDict
assert cloned == original
assert cloned.data is not original.data

cloned["x"] = 9

assert original["x"] == 1
assert cloned["x"] == 9

print("Original:", original)
print("Clone:", cloned)


Original: {'x': 1, 'y': 2}
Clone: {'x': 9, 'y': 2}


## Problem 15 — Interoperability: views, conversion, and snapshots

Using `StrictIntViewDict`, inspect:

- `keys()`,
- `values()`,
- `items()`,
- `dict(d)`,
- `.data`,
- a deep raw snapshot.

Write `public_snapshot(mapping)` that captures public lookup semantics.


### Solution 15


In [16]:
d3 = StrictIntViewDict(a=1.9, b=2.1, c=-3.8)

print("keys:", list(d3.keys()))
print("values:", list(d3.values()))
print("items:", list(d3.items()))
print("dict(d3):", dict(d3))
print("raw .data:", d3.data)


def public_snapshot(mapping):
    return {key: mapping[key] for key in mapping}


public = public_snapshot(d3)
raw = d3.data.copy()
deep_raw = deepcopy(d3.data)

assert public == {"a": 1, "b": 2, "c": -3}
assert raw == {"a": 1.9, "b": 2.1, "c": -3.8}
assert deep_raw == raw
assert deep_raw is not d3.data

print("Public snapshot:", public)
print("Raw snapshot:", raw)


keys: ['a', 'b', 'c']
values: [1, 2, -3]
items: [('a', 1), ('b', 2), ('c', -3)]
dict(d3): {'a': 1, 'b': 2, 'c': -3}
raw .data: {'a': 1.9, 'b': 2.1, 'c': -3.8}
Public snapshot: {'a': 1, 'b': 2, 'c': -3}
Raw snapshot: {'a': 1.9, 'b': 2.1, 'c': -3.8}


A public snapshot answers "what values do callers observe?" while a raw snapshot answers "what is physically stored?" Those can intentionally differ when reads transform values.


# Problem 16 — Capstone: transactional configuration registry

Build `ConfigRegistry(UserDict)` with:

1. schema validation/coercion,
2. atomic construction and update,
3. required keys,
4. write-once fields,
5. audit logging,
6. detached snapshots.

Use the earlier validators:

- `host`: stripped non-empty string
- `port`: integer `1..65535`
- `debug`: exact `bool`
- `timeout`: positive finite real -> `float`

Required keys: `host`, `port`.

Write-once key: `host`.


## Solution 16


In [17]:
class ConfigRegistry(UserDict):
    def __init__(
        self,
        schema,
        *,
        required=(),
        write_once=(),
        initial=None,
        **kwargs,
    ):
        self._schema = dict(schema)
        self._required = frozenset(required)
        self._write_once = frozenset(write_once)
        self.audit_log = []

        unknown_required = self._required - self._schema.keys()
        unknown_write_once = self._write_once - self._schema.keys()

        if unknown_required:
            raise ValueError(
                f"required keys missing from schema: {sorted(unknown_required)!r}"
            )
        if unknown_write_once:
            raise ValueError(
                f"write-once keys missing from schema: {sorted(unknown_write_once)!r}"
            )

        super().__init__()

        if initial is not None or kwargs:
            self.update(initial or {}, **kwargs)

    def _coerce(self, key, value):
        try:
            validator = self._schema[key]
        except KeyError:
            raise KeyError(
                f"unsupported configuration field: {key!r}"
            ) from None
        return validator(value)

    def __setitem__(self, key, value):
        self.update({key: value})

    def update(self, other=None, **kwargs):
        candidates = {}

        if other is not None:
            if isinstance(other, Mapping) or hasattr(other, "keys"):
                for key in other.keys():
                    candidates[key] = other[key]
            else:
                for key, value in other:
                    candidates[key] = value

        candidates.update(kwargs)

        staged = {}

        for key, raw_value in candidates.items():
            if key in self._write_once and key in self.data:
                raise KeyError(f"field {key!r} is write-once")
            staged[key] = self._coerce(key, raw_value)

        for key, new_value in staged.items():
            old_value = self.data.get(key, _MISSING)
            self.data[key] = new_value
            self.audit_log.append(("set", key, old_value, new_value))

    def validate_complete(self):
        missing = self._required - self.data.keys()
        if missing:
            raise ValueError(
                f"missing required configuration: {sorted(missing)!r}"
            )
        return True

    def snapshot(self):
        return deepcopy(self.data)


registry = ConfigRegistry(
    schema,
    required={"host", "port"},
    write_once={"host"},
    initial={"host": "  example.com  ", "port": 443},
    debug=False,
    timeout=2,
)

assert registry.validate_complete()
before = registry.snapshot()

expect_exception(
    ValueError,
    registry.update,
    {"debug": True, "timeout": -1},
)

assert registry.snapshot() == before

registry.update({"debug": True, "timeout": 5})
assert registry["debug"] is True
assert registry["timeout"] == 5.0

expect_exception(
    KeyError,
    registry.__setitem__,
    "host",
    "other.example.com",
)
expect_exception(
    KeyError,
    registry.__setitem__,
    "unknown",
    123,
)

snapshot = registry.snapshot()
snapshot["port"] = 80

assert registry["port"] == 443
assert snapshot["port"] == 80

print("Final registry:", registry)
print("Detached snapshot:", snapshot)
print("Audit log:")
for operation, key, old, new in registry.audit_log:
    old_display = "<MISSING>" if old is _MISSING else old
    print(f"  {operation:>3} {key!r}: {old_display!r} -> {new!r}")


Expected ValueError: timeout must be positive and finite
Expected KeyError: "field 'host' is write-once"
Expected KeyError: "unsupported configuration field: 'unknown'"
Final registry: {'host': 'example.com', 'port': 443, 'debug': True, 'timeout': 5.0}
Detached snapshot: {'host': 'example.com', 'port': 80, 'debug': True, 'timeout': 5.0}
Audit log:
  set 'host': '<MISSING>' -> 'example.com'
  set 'port': '<MISSING>' -> 443
  set 'debug': '<MISSING>' -> False
  set 'timeout': '<MISSING>' -> 2.0
  set 'debug': False -> True
  set 'timeout': 2.0 -> 5.0


## Capstone extension challenges

1. Add a `resettable` set of keys that may be deleted.
2. Add `freeze()` so no field can change after freezing.
3. Add `diff(other_mapping)` returning added, removed, and changed keys.
4. Add `rollback(n=1)` using the audit log.
5. Add per-field documentation plus `describe()`.
6. Add a context-manager transaction.
7. Add nested schema mappings.
8. Add `to_json_ready()`.
9. Add strict-vs-coercing validation modes.
10. Write randomized valid/invalid update tests.


# Bonus Problem A — Predict before running

Predict:

```python
x = StrictIntViewDict(a=9.9)
x.update(b=3.2)
print(x["a"])
print(x.get("b"))
print(x.data)
print(dict(x))
```


## Bonus Solution A


In [18]:
x = StrictIntViewDict(a=9.9)
x.update(b=3.2)

print(x["a"])
print(x.get("b"))
print(x.data)
print(dict(x))

assert x.data == {"a": 9.9, "b": 3.2}
assert dict(x) == {"a": 9, "b": 3}


9
3
{'a': 9.9, 'b': 3.2}
{'a': 9, 'b': 3}


# Bonus Problem B — Find the bug

Review:

```python
class BadRangeDict(UserDict):
    def __setitem__(self, key, value):
        if not isinstance(value, int):
            raise TypeError
        if value < 0 or value > 10:
            raise ValueError
        self.data[key] = value
```

Find at least three weaknesses.

## Bonus Solution B

Possible weaknesses:

1. `bool` passes `isinstance(value, int)`.
2. Bare exceptions provide poor diagnostics.
3. `super().__setitem__` communicates intent better than writing `.data` directly.
4. The range is hard-coded.
5. No constructor/update/setdefault tests exist.
6. `.data` can still bypass validation.


In [19]:
class BetterRangeDict(UserDict):
    def __init__(self, low=0, high=10, *args, **kwargs):
        if low > high:
            raise ValueError("low cannot exceed high")
        self.low = low
        self.high = high
        super().__init__(*args, **kwargs)

    def __setitem__(self, key, value):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError("value must be an integer, not bool")
        if not self.low <= value <= self.high:
            raise ValueError(f"value must be in [{self.low}, {self.high}]")
        super().__setitem__(key, value)


r = BetterRangeDict(-5, 5, a=0)
r.update(b=5)
expect_exception(ValueError, r.__setitem__, "c", 6)
expect_exception(TypeError, r.__setitem__, "c", True)
print(r)


Expected ValueError: value must be in [-5, 5]
Expected TypeError: value must be an integer, not bool
{'a': 0, 'b': 5}


# Bonus Problem C — API test matrix

A strong custom mapping should be tested through more than direct assignment.

Exercise:

- direct assignment,
- update from mapping,
- update from iterable,
- keyword update,
- `setdefault`,
- `get`,
- views,
- `pop`,
- plain-dict conversion.


In [20]:
def exercise_mapping_api(factory):
    m = factory()

    m["a"] = 1
    m.update({"b": 2})
    m.update([("c", 3)])
    m.update(d=4)

    assert m.setdefault("a", 99) == 1
    assert m.get("b") == 2
    assert set(m.keys()) == {"a", "b", "c", "d"}
    assert dict(m.items()) == {"a": 1, "b": 2, "c": 3, "d": 4}
    assert m.pop("d") == 4

    plain = dict(m)
    assert plain == {"a": 1, "b": 2, "c": 3}

    return m


tested = exercise_mapping_api(
    lambda: LimitedDict({"a", "b", "c", "d"}, 0, 10)
)

print("API matrix passed:", tested)


API matrix passed: {'a': 1, 'b': 2, 'c': 3}


# Final review questions

1. Why can subclassing `dict` be surprising when overriding `__getitem__` and `__setitem__`?
2. What problem does `UserDict` solve?
3. What is stored in `.data`?
4. Why should validation usually be centralized?
5. Why is `bool` a numeric-validation edge case?
6. What is the difference between `TypeError` and `ValueError`?
7. How can a failed multi-item `update` cause partial mutation?
8. How does an atomic update prevent that?
9. When is read-time transformation useful?
10. Why might `dict(custom_mapping)` differ from `custom_mapping.data.copy()`?
11. Why can `.data` undermine invariants?
12. When might `MutableMapping` with private storage be preferable?
13. How should configured subclasses implement copying?
14. Why should constructor behavior be tested explicitly?
15. Which exercise most resembles a production configuration system?


# Summary

The key principle is not only "inherit from `UserDict`." It is:

> Define your mapping's invariants and public semantics first, then make every supported operation preserve them.

`UserDict` is useful because it is a Python-level mapping implementation designed for customization, so construction, lookup, update, and inherited mapping methods cooperate more naturally with overrides.

For advanced code, also consider atomicity, coercion policy, internal-vs-public representation, copying, logging, and the fact that `.data` is an escape hatch around validation.
